In [1]:
# ============================================
# STEP 1 — LOAD LIBRARIES & DESCRIBE SCRIPT
# ============================================

"""
SCRIPT OVERVIEW

This script calculates fabric upgrade capital costs for each dwelling in an EPC dataset.

Inputs:
- CSV file containing building-level data
- Required columns:
    WALL_AREA, WINDOW_AREA, ROOF_AREA, FOOTPRINT_AREA
    WALLS_UPGRADE_TYPE, WINDOWS_UPGRADE_TYPE, ROOF_UPGRADE_TYPE

Outputs:
- Updated columns:
    WALLS_CAP_COST
    WINDOWS_CAP_COST
    ROOF_CAP_COST
    FAB_CAP_COST (sum of all above)

Cost logic:
- Walls:
    none → 0
    external → WALL_AREA * 150
    external_and_cavity → WALL_AREA * 200
    cavity → WALL_AREA * 50
    internal → WALL_AREA * 120

- Windows:
    uPVC → WINDOW_AREA * 700

- Roof:
    loft → FOOTPRINT_AREA * 20
    rafters → ROOF_AREA * 20

- Fabric total:
    FAB_CAP_COST = sum of all components

This script is structured in steps for debugging and reuse.
"""

import pandas as pd
import numpy as np

In [2]:
# ============================================
# STEP 2 — LOAD DATA
# ============================================

file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_wall_area_update_2.csv"

df = pd.read_csv(file_path)

# Preview dataset
df.head()

,BUILDING_REFERENCE_NUMBER,OSG_REFERENCE_NUMBER,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,LATITUDE,LONGITUDE,ORIENTATION,ORIENTATION_ESPR_ROT,...,BATTERY_NPV_FLEX,WINDOWS_IRR_FLEX,ROOF_IRR_FLEX,WALLS_IRR_FLEX,FAB_IRR_FLEX,HP_IRR_FLEX,SOLAR_THERMAL_IRR_FLEX,PV_IRR_FLEX,BATTERY_IRR_FLEX,PEAK_HP_LOAD
0,1001829067,122009933.0,19 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.0557,-4.2233,100,170,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,977.038423
1,1001951858,122010025.0,GLENVIEW,FINTRY,GLASGOW,G63 0XJ,56.0528,-4.2206,180,90,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2363.181338
2,1001829071,122009868.0,22 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.0555,-4.2237,100,170,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,540.540497
3,1234761001,122009968.0,1 MENZIES TERRACE,FINTRY,GLASGOW,G63 0YJ,56.0584,-4.2248,150,120,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1431.249048
4,1001991633,122009892.0,50 MAIN STREET,FINTRY,GLASGOW,G63 0XF,56.0547,-4.2283,150,120,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,473.933683


In [3]:
# ============================================
# STEP 3 — CLEAN COLUMN NAMES (IMPORTANT)
# ============================================

# Standardise column names (remove spaces, uppercase)
df.columns = df.columns.str.strip().str.upper().str.replace(" ", "_")

# Fix known issue if column exists with typo
if "FOOT_PRINT_AREA" in df.columns:
    df.rename(columns={"FOOT_PRINT_AREA": "FOOTPRINT_AREA"}, inplace=True)

df.columns

Index(['BUILDING_REFERENCE_NUMBER', 'OSG_REFERENCE_NUMBER', 'ADDRESS1',
       'ADDRESS2', 'ADDRESS3', 'POSTCODE', 'LATITUDE', 'LONGITUDE',
       'ORIENTATION', 'ORIENTATION_ESPR_ROT',
       ...
       'BATTERY_NPV_FLEX', 'WINDOWS_IRR_FLEX', 'ROOF_IRR_FLEX',
       'WALLS_IRR_FLEX', 'FAB_IRR_FLEX', 'HP_IRR_FLEX',
       'SOLAR_THERMAL_IRR_FLEX', 'PV_IRR_FLEX', 'BATTERY_IRR_FLEX',
       'PEAK_HP_LOAD'],
      dtype='object', length=320)

In [4]:
# ============================================
# STEP 4 — CALCULATE WALLS_CAP_COST
# ============================================

def calc_walls_cost(row):
    wall_type = str(row["WALLS_UPGRADE_TYPE"]).lower()
    area = row["WALL_AREA"]

    if wall_type == "none":
        return 0
    elif wall_type == "external":
        return area * 150
    elif wall_type == "external_and_cavity":
        return area * 200
    elif wall_type == "cavity":
        return area * 50
    elif wall_type == "internal":
        return area * 120
    else:
        return 0  # fallback for missing/unknown

df["WALLS_CAP_COST"] = df.apply(calc_walls_cost, axis=1)

df[["WALLS_UPGRADE_TYPE", "WALL_AREA", "WALLS_CAP_COST"]].head()

,WALLS_UPGRADE_TYPE,WALL_AREA,WALLS_CAP_COST
0,external,65.989090,9898.363501
1,external,155.537777,23330.666514
2,external,NaN,NaN
3,external_and_cavity,156.153770,31230.754074
4,external,NaN,NaN


In [5]:
# ============================================
# STEP 5 — CALCULATE WINDOWS_CAP_COST
# ============================================

def calc_windows_cost(row):
    window_type = str(row["WINDOWS_UPGRADE_TYPE"]).lower()
    area = row["WINDOW_AREA"]

    if window_type == "upvc":
        return area * 700
    else:
        return 0

df["WINDOWS_CAP_COST"] = df.apply(calc_windows_cost, axis=1)

df[["WINDOWS_UPGRADE_TYPE", "WINDOW_AREA", "WINDOWS_CAP_COST"]].head()

,WINDOWS_UPGRADE_TYPE,WINDOW_AREA,WINDOWS_CAP_COST
0,uPVC,12.5358,8775.06
1,uPVC,22.3276,15629.32
2,uPVC,14.3142,10019.94
3,uPVC,23.0673,16147.11
4,uPVC,19.3444,13541.08


In [6]:
# ============================================
# STEP 6 — CALCULATE ROOF_CAP_COST
# ============================================

def calc_roof_cost(row):
    roof_type = str(row["ROOF_UPGRADE_TYPE"]).lower()
    
    if roof_type == "loft":
        return row["FOOTPRINT_AREA"] * 20
    elif roof_type == "rafters":
        return row["ROOF_AREA"] * 20
    else:
        return 0

df["ROOF_CAP_COST"] = df.apply(calc_roof_cost, axis=1)

df[["ROOF_UPGRADE_TYPE", "ROOF_AREA", "FOOTPRINT_AREA", "ROOF_CAP_COST"]].head()

,ROOF_UPGRADE_TYPE,ROOF_AREA,FOOTPRINT_AREA,ROOF_CAP_COST
0,loft,29.698485,21.0,420.0
1,loft,89.095454,63.0,1260.0
2,loft,48.083261,34.0,680.0
3,loft,89.802561,63.5,1270.0
4,loft,194.155227,104.0,2080.0


In [7]:
# ============================================
# STEP 7 — CALCULATE FAB_CAP_COST
# ============================================

df["FAB_CAP_COST"] = (
    df["WALLS_CAP_COST"] +
    df["WINDOWS_CAP_COST"] +
    df["ROOF_CAP_COST"]
)

df[["WALLS_CAP_COST", "WINDOWS_CAP_COST", "ROOF_CAP_COST", "FAB_CAP_COST"]].head()

,WALLS_CAP_COST,WINDOWS_CAP_COST,ROOF_CAP_COST,FAB_CAP_COST
0,9898.363501,8775.06,420.0,19093.423501
1,23330.666514,15629.32,1260.0,40219.986514
2,NaN,10019.94,680.0,NaN
3,31230.754074,16147.11,1270.0,48647.864074
4,NaN,13541.08,2080.0,NaN


In [8]:
# ============================================
# STEP 8 — SAVE OUTPUT FILE
# ============================================

output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_with_costs_2.csv"

df.to_csv(output_path, index=False)

print(f"Updated dataset saved to: {output_path}")

Updated dataset saved to: /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_with_costs_2.csv
